# Weight Cap 실험 — x_max ∈ {0.3, 0.6}

자산별 비중 상한을 걸었을 때 성과가 어떻게 달라지는지 본다.

- 학습·병합은 `run_xmax_sweep.py` 가 전담 (x_max=0.3 완주 → 0.6 순차)
- 체크포인트는 `_xm0.3` / `_xm0.6` 태그로 기존(x_max=1.0) 결과와 분리
- 분석 로직은 기존 `.py` 모듈 재사용 — 이 노트북에는 호출만 둔다

| 파일명 예시 | 의미 |
| --- | --- |
| `dfl_mdd_30_inds_h126_d20_l0.5_CLARABEL.pkl` | x_max = 1.0 (기존) |
| `dfl_mdd_30_inds_h126_xm0.3_d20_l0.5_CLARABEL.pkl` | x_max = 0.3 |

## 1. 설정

In [2]:
import os, sys, pickle, subprocess, time, importlib, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

import benchmarks, performance, carryforward
for _m in (benchmarks, performance, carryforward):
    importlib.reload(_m)

from benchmarks import (build_bench_store, attach_date_idx,
                        compute_turnover, rebalance_dates)
from carryforward import apply_carryforward, parse_lb, parse_n1
from performance import compute_performance, apply_tc, build_equity_curve

# ── 실험 설정 ──
N_STOCKS      = 10
HORIZON       = 126
REBAL         = 21
SOLVER        = "CLARABEL"
DELTA         = 20
LAM_LIST      = [0.3, 0.5, 0.7, 1.0]
LOOKBACK_LIST = [252, 504]
N1_LIST       = [0.1, 0.2, 0.3, 0.4]
XMAX_LIST     = [0.3, 0.6]           # 이번 실험
XMAX_ALL      = XMAX_LIST + [1.0]    # 비교 대상 (1.0 = 기존 결과)

VAL_YEARS, TEST_YEARS, N_FOLDS = 5, 1, 8
CKPT_DIR   = "./checkpoint"
PLOT_DIR   = "./plots/xmax"
RESULT_DIR = "./results"
for _d in (CKPT_DIR, PLOT_DIR, RESULT_DIR):
    os.makedirs(_d, exist_ok=True)

configs = [{"LOOKBACK": lb, "n1": n1} for lb in LOOKBACK_LIST for n1 in N1_LIST]
print(f"{N_STOCKS} inds, H={HORIZON}, x_max {XMAX_ALL}, config {len(configs)}개")

10 inds, H=126, x_max [0.3, 0.6, 1.0], config 8개


## 2. 데이터 · fold

In [3]:
inds = pd.read_csv(f"csv/{N_STOCKS}_industry.csv")
inds["Date"] = pd.to_datetime(inds["Date"])
inds = inds.set_index("Date").sort_index()
inds = inds[~inds.index.duplicated(keep="first")] / 100.0

stock_names = inds.columns.tolist()
full_np     = inds.values
full_dates  = inds.index
d, C_cap    = 1.0, 1.0

def date_to_idx(s):
    return full_dates.searchsorted(pd.Timestamp(s), side="left")

def make_folds(horizon):
    fs = []
    for f in range(N_FOLDS):
        ty = 2018 + f
        fs.append({
            "fold"          : f + 1,
            "train_end_idx" : date_to_idx(f"{ty - VAL_YEARS}-01-01"),
            "val_start_idx" : date_to_idx(f"{ty - VAL_YEARS}-01-01"),
            "val_end_idx"   : date_to_idx(f"{ty}-01-01"),
            "test_start_idx": date_to_idx(f"{ty}-01-01"),
            "test_end_idx"  : min(date_to_idx(f"{ty + TEST_YEARS}-01-01") + horizon,
                                  len(full_np)),
            "val_year"      : f"{ty - VAL_YEARS}~{ty - 1}",
            "test_year"     : ty,
        })
    return fs

folds = make_folds(HORIZON)
N_WIN = sum(len(rebalance_dates(f, 252, HORIZON, REBAL)) for f in folds)
print(f"{len(full_np)}일, {full_dates[0]:%Y-%m-%d} ~ {full_dates[-1]:%Y-%m-%d}")
print(f"fold {len(folds)}개, config당 리밸런싱 윈도우 {N_WIN}개")

6539일, 2000-01-03 ~ 2025-12-31
fold 8개, config당 리밸런싱 윈도우 94개


## 3. 학습 실행

`run_xmax_sweep.py` 가 **x_max=0.3 전체(DFL-MDD 16 shard → 병합 → DFL-MVO 4 shard)를
끝낸 뒤 0.6 을 시작**한다. 0.6 이 도는 동안 0.3 결과로 아래 분석을 진행할 수 있다.

예상 소요: x_max 하나당 DFL-MDD ~8시간 + DFL-MVO ~3시간 → 전체 ~22시간

In [ ]:
PY  = sys.executable
ENV = {**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUTF8": "1"}
os.makedirs("./logs", exist_ok=True)

import psutil
free = psutil.virtual_memory().available / 1e9
JOBS = max(1, min(16, int((free - 3.0) / 1.40)))   # 워커당 ~1.4GB
print(f"가용 RAM {free:.1f} GB → DFL-MDD 동시 {JOBS}개")

sweep_log = f"./logs/xmax_sweep_{N_STOCKS}_h{HORIZON}.txt"
f_sweep = open(sweep_log, "w", encoding="utf-8")
proc_sweep = subprocess.Popen(
    [PY, "-u", "run_xmax_sweep.py",
     "--data", str(N_STOCKS), "--horizon", str(HORIZON),
     "--delta", str(DELTA), "--solver", SOLVER,
     "--xmax", *[f"{x:g}" for x in XMAX_LIST],
     "--jobs", str(JOBS), "--python", PY],
    stdout=f_sweep, stderr=subprocess.STDOUT, env=ENV)

print(f"sweep 시작 — x_max {XMAX_LIST} 순차 (PID {proc_sweep.pid})")
print(f"  진행 로그: {sweep_log}")

### 3-1. 진행 확인 (반복 실행)

In [ ]:
rc = proc_sweep.poll()
print("실행 중" if rc is None else f"sweep 종료 (rc={rc})")
print(open(sweep_log, encoding="utf-8").read()[-1000:])

PER     = len(LOOKBACK_LIST) * len(folds)          # shard(lam,n1) 당 유닛 = 16
GRAND   = len(LAM_LIST) * len(N1_LIST) * PER       # 전체 = 256
DONE_RE = re.compile(r"n1=[\d.]+\s+(?:fallback|리밸런싱)")
SKIP_RE = re.compile(r"fold \d+ 스킵")
EL_RE   = re.compile(r"\+(\d+):(\d{2}):(\d{2})\]")

def fmt(s):
    s = int(max(s, 0))
    return f"{s//3600}:{(s%3600)//60:02d}:{s%60:02d}"

print("\n── DFL-MDD 진행 ──")
print(f"{'x_max':>7}{'진행':>11}{'경과':>11}{'잔여(추정)':>12}"
      f"{'fallback':>18}{'수치실패':>10}  상태")

for xm in XMAX_LIST:
    tag  = f"_xm{xm:g}"
    done = elapsed = 0
    started = False

    # ① 진행·경과 — 로그에서
    for lam in LAM_LIST:
        for n1 in N1_LIST:
            for lb in LOOKBACK_LIST:
                p = (f"./logs/mdd_{N_STOCKS}_h{HORIZON}{tag}"
                     f"_LB{lb}_n1{n1:g}_l{lam}.txt")
                if not os.path.exists(p): continue
                started = True
                t = open(p, encoding="utf-8", errors="replace").read()
                done += len(DONE_RE.findall(t)) + len(SKIP_RE.findall(t))
                e = EL_RE.findall(t)
                if e:
                    elapsed = max(elapsed,
                                  int(e[-1][0])*3600 + int(e[-1][1])*60 + int(e[-1][2]))

    # ② infeasible — shard 체크포인트에서 (재개해도 누적되므로 정확)
    n_inf = n_win = n_num = 0
    for lam in LAM_LIST:
        for n1 in N1_LIST:
            for lb in LOOKBACK_LIST:
                cp = (f"{CKPT_DIR}/dfl_mdd_{N_STOCKS}_inds_h{HORIZON}{tag}"
                      f"_LB{lb}_n1{n1:g}_d{DELTA}_l{lam}_{SOLVER}.pkl")
                if not os.path.exists(cp): continue
                try:
                    with open(cp, "rb") as f: ck = pickle.load(f)
                except Exception:
                    continue
                for v in ck.get("infeas_map", {}).values():
                    for e in v:
                        n_inf += e["n_infeasible"]
                        n_win += e["n_windows"]
                        n_num += e.get("n_numerical", 0)

    eta   = fmt(elapsed / done * (GRAND - done)) if done else "—"
    rate  = f"{n_inf}/{n_win} ({n_inf/n_win:.1%})" if n_win else "—"
    state = "대기 중" if not started else ("완료" if done >= GRAND else "진행 중")
    print(f"{xm:>7g}{f'{done}/{GRAND}':>11}{fmt(elapsed):>11}{eta:>12}"
          f"{rate:>18}{n_num:>10}  {state}")

print(f"\n  fallback = 최적화 실패 → carry-forward (직전 비중 유지, 첫 윈도우는 EW)")
print(f"  수치실패 > 0 이면 solver 문제 — 확인 필요")

## 4. 결과 로드

`x_max` 별로 DFL-MDD(+carry-forward), DFL-MVO, 벤치마크를 한 번에 읽는다.
`x_max=1.0` 은 기존 체크포인트(태그 없음)를 그대로 사용한다.

**벤치마크에도 동일한 상한을 적용**한다 (`build_bench_store(..., x_max=xm)`).

In [ ]:
def xm_tag(xm):
    return "" if xm >= 1.0 else f"_xm{xm:g}"

def load_xmax(xm, verbose=True):
    """x_max 하나에 대한 DFL-MDD(cf) / DFL-MVO / 벤치마크를 모두 로드."""
    tag = xm_tag(xm)
    store, infs = {}, {}
    for lam in LAM_LIST:
        p = (f"{CKPT_DIR}/dfl_mdd_{N_STOCKS}_inds_h{HORIZON}{tag}"
             f"_d{DELTA}_l{lam}_{SOLVER}.pkl")
        if not os.path.exists(p):
            if verbose: print(f"  ! 없음: {os.path.basename(p)}")
            continue
        with open(p, "rb") as f: ck = pickle.load(f)
        store[(DELTA, lam)] = [(ck["fold_results_map"][(c["LOOKBACK"], c["n1"])],
                                f"DFL-MDD (LB={c['LOOKBACK']}, n1={c['n1']})")
                               for c in configs]
        infs[(DELTA, lam)] = ck["infeas_map"]
    if not store:
        return None

    cf = apply_carryforward(store, infs, folds=folds, full_np=full_np,
                            HORIZON=HORIZON, REBAL=REBAL, d=d, C=C_cap,
                            verbose=False)
    cf = {k: [(attach_date_idx(r, folds, parse_lb(l), HORIZON, REBAL), l)
              for r, l in v] for k, v in cf.items()}

    mvo = {}
    for lam in LAM_LIST:
        p = (f"{CKPT_DIR}/dfl_mvo_{N_STOCKS}_inds_h{HORIZON}{tag}"
             f"_d{DELTA}_l{lam}_{SOLVER}.pkl")
        if not os.path.exists(p): continue
        with open(p, "rb") as f: ck = pickle.load(f)
        mvo[(DELTA, lam)] = [(attach_date_idx(ck["fold_results_map"][lb], folds,
                                              lb, HORIZON, REBAL),
                              f"DFL-MVO (LB={lb})") for lb in LOOKBACK_LIST]

    bench, _ = build_bench_store(full_np, folds, stock_names, LOOKBACK_LIST,
                                 HORIZON, REBAL, delta=DELTA, x_max=xm,
                                 verbose=False)
    if verbose:
        w = np.concatenate([[r["weights"] for r in res]
                            for res, _ in cf[(DELTA, LAM_LIST[0])]])
        n = len(cf[(DELTA, LAM_LIST[0])][0][0])
        print(f"  x_max={xm:<4g} config {len(cf[(DELTA, LAM_LIST[0])])}개, "
              f"윈도우 {n}, 최대비중 {w.max():.4f}, "
              f"DFL-MVO {len(mvo)}개 λ, 벤치 {len(bench)}종")
        if w.max() > xm + 1e-6:
            print(f"     ⚠ 상한 위반! {w.max():.4f} > {xm}")
    return {"cf": cf, "mvo": mvo, "bench": bench}

DATA = {}
for xm in XMAX_ALL:
    r = load_xmax(xm)
    if r: DATA[xm] = r
print(f"\n로드된 x_max: {sorted(DATA)}")

## 5. 성과표 — x_max 비교

In [ ]:
rows = []
for xm, D in sorted(DATA.items()):
    for lam in LAM_LIST:
        if (DELTA, lam) not in D["cf"]: continue
        groups = [("DFL-MDD",   D["cf"][(DELTA, lam)]),
                  ("DFL-MVO",   D["mvo"].get((DELTA, lam), [])),
                  ("Benchmark", [(v, k) for k, v in D["bench"].items()])]
        for g, items in groups:
            for res, lbl in items:
                p  = compute_performance(res)
                to = compute_turnover(res, full_np, REBAL)["mean"]
                w  = np.array([r["weights"] for r in res], dtype=float)
                rows.append({"x_max": xm, "lam": lam, "group": g, "label": lbl,
                             "Ann.Ret(%)": round(p["Ann.Ret"] * 100, 2),
                             "Sharpe": round(p["Sharpe"], 4),
                             "MDD(%)": round(p["MDD"] * 100, 2),
                             "MDD_abs(%)": round(p["MDD_abs"] * 100, 2),
                             "Calmar": round(p["Calmar"], 4),
                             "HHI": round(p["HHI"], 4),
                             "MaxW": round(float(w.max()), 4),
                             "nActive": round(float((w > 1e-4).sum(1).mean()), 2),
                             "Turnover": round(to, 4)})
df_xm = pd.DataFrame(rows)
out = f"{RESULT_DIR}/{N_STOCKS}_inds_h{HORIZON}_xmax_compare.csv"
df_xm.to_csv(out, index=False, encoding="utf-8-sig")
print(f"✓ 저장: {out}  ({len(df_xm)}행)")

print("\n── DFL-MDD 그룹 평균 (config 8개) ──")
display(df_xm[df_xm.group == "DFL-MDD"].groupby(["x_max", "lam"])[
    ["Ann.Ret(%)", "Sharpe", "MDD(%)", "Calmar", "HHI", "nActive", "Turnover"]
].mean().round(3))

### 5-1. 집중도 변화 — 상한이 실제로 작동했는가

In [ ]:
display(df_xm[df_xm.group == "DFL-MDD"].pivot_table(
    index="lam", columns="x_max", values=["HHI", "nActive", "MaxW"]).round(3))

viol = df_xm[df_xm.MaxW > df_xm.x_max + 1e-6]
print(f"상한 위반: {len(viol)}건" + ("" if len(viol) == 0 else " ← 확인 필요"))
if len(viol):
    display(viol[["x_max", "lam", "group", "label", "MaxW"]])

## 6. 거래비용 반영

In [ ]:
TCS = [0, 5, 10, 20, 40]
rows = []
for xm, D in sorted(DATA.items()):
    for lam in LAM_LIST:
        if (DELTA, lam) not in D["cf"]: continue
        groups = [("DFL-MDD",   D["cf"][(DELTA, lam)]),
                  ("DFL-MVO",   D["mvo"].get((DELTA, lam), [])),
                  ("Benchmark", [(v, k) for k, v in D["bench"].items()])]
        for tc in TCS:
            for g, items in groups:
                for res, lbl in items:
                    r2 = apply_tc(res, tc / 1e4, full_np, REBAL) if tc else res
                    p  = compute_performance(r2)
                    rows.append({"x_max": xm, "lam": lam, "tc_bps": tc,
                                 "group": g, "label": lbl,
                                 "Ann.Ret(%)": round(p["Ann.Ret"] * 100, 2),
                                 "Sharpe": round(p["Sharpe"], 4),
                                 "MDD(%)": round(p["MDD"] * 100, 2),
                                 "Calmar": round(p["Calmar"], 4)})
df_tc = pd.DataFrame(rows)
out = f"{RESULT_DIR}/{N_STOCKS}_inds_h{HORIZON}_xmax_tc.csv"
df_tc.to_csv(out, index=False, encoding="utf-8-sig")
print(f"✓ 저장: {out}  ({len(df_tc)}행)")

print("\n── Calmar: tc × x_max (DFL-MDD 평균) ──")
display(df_tc[df_tc.group == "DFL-MDD"].pivot_table(
    index="tc_bps", columns="x_max", values="Calmar").round(4))
print("── MDD(%): tc × x_max (DFL-MDD 평균) ──")
display(df_tc[df_tc.group == "DFL-MDD"].pivot_table(
    index="tc_bps", columns="x_max", values="MDD(%)").round(2))

## 7. 누적수익 비교

In [ ]:
XM_COL = {1.0: "#9AA5B1", 0.6: "#0B6E8F", 0.3: "#B23A48"}

for lam in LAM_LIST:
    avail = [xm for xm in sorted(DATA) if (DELTA, lam) in DATA[xm]["cf"]]
    if not avail: continue
    fig, axes = plt.subplots(1, len(LOOKBACK_LIST), figsize=(13, 4.4), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, lb in zip(axes, LOOKBACK_LIST):
        for xm in avail:
            sel = [res for res, l in DATA[xm]["cf"][(DELTA, lam)] if parse_lb(l) == lb]
            if not sel: continue
            eqs = [build_equity_curve(r) for r in sel]
            L   = min(len(e) for e in eqs)
            eq  = np.mean([e[:L] for e in eqs], axis=0)      # n1 4개 평균
            mdd = np.mean([compute_performance(r)["MDD"] for r in sel])
            cal = np.mean([compute_performance(r)["Calmar"] for r in sel])
            ax.plot(eq, color=XM_COL.get(xm, "#666"), lw=1.7,
                    label=f"$x_{{max}}$={xm:g}   MDD {mdd:.1%}  Cal {cal:.2f}")
        ax.axhline(1.0, color="gray", ls="--", lw=0.8, alpha=0.6)
        ax.set_title(f"Lookback = {lb}", fontsize=11.5)
        ax.set_xlabel("Trading days (test period)")
        ax.legend(fontsize=8, loc="upper left")
        ax.grid(alpha=0.22, lw=0.7)
        for sp in ("top", "right"): ax.spines[sp].set_visible(False)
    axes[0].set_ylabel("Portfolio value ($n_1$ 평균)")
    fig.suptitle(f"Weight cap 비교  (H={HORIZON}, $\\lambda$={lam})",
                 fontsize=13, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    out = f"{PLOT_DIR}/cumret_xmax_{N_STOCKS}_inds_h{HORIZON}_lam{lam}.png"
    plt.savefig(out, bbox_inches="tight", dpi=300)
    plt.show()
    print(f"  ✓ {out}")

## 8. 통계검정

`x_max` 별로 기존과 같은 단측 대응 t-검정을 돌린다.
DFL-MDD·DFL-MVO·벤치마크가 모두 같은 상한을 쓰므로 공정 비교가 된다.

In [ ]:
from scipy import stats

ALPHAS = [0.10, 0.05, 0.01]

def sig(p1, a):
    if not np.isfinite(p1): return "n/a"
    return "✓" if p1 < a else ("✗" if p1 > 1 - a else "–")

def ttest_for(xm):
    D = DATA[xm]
    def dates_of(st): return {r["date_idx"] for res, _ in st for r in res}
    sets = {"mdd": dates_of(D["cf"][(DELTA, LAM_LIST[0])]),
            "mvo": dates_of(D["mvo"][(DELTA, LAM_LIST[0])])}
    for l, r in D["bench"].items():
        sets[l] = {x["date_idx"] for x in r}
    COMMON = sorted(set.intersection(*sets.values()))

    def per_date(st):
        rec = {}
        for res, _ in st:
            for r in res:
                rec.setdefault(r["date_idx"], []).append(r["M_real"] * 100)
        return pd.Series({k: np.mean(v) for k, v in rec.items()}).loc[COMMON]

    def by_lb(st, lb): return [(r, l) for r, l in st if parse_lb(l) == lb]

    out = []
    for lb in LOOKBACK_LIST:
        for lam in LAM_LIST:
            base  = per_date(by_lb(D["cf"][(DELTA, lam)], lb))
            comps = {"DFL-MVO":  by_lb(D["mvo"][(DELTA, lam)], lb),
                     "GMV":      [(D["bench"][f"GMV (LB={lb})"], "")],
                     "hist-MVO": [(D["bench"][f"hist-MVO (LB={lb})"], "")],
                     "EW":       [(D["bench"]["EW"], "")]}
            for nm, st in comps.items():
                x, y = base.values, per_date(st).values
                diff = x - y
                t, p2 = stats.ttest_rel(x, y)
                p1 = p2 / 2 if t < 0 else 1 - p2 / 2
                try:
                    _, w2 = stats.wilcoxon(x, y)
                    w1 = w2 / 2 if np.median(diff) < 0 else 1 - w2 / 2
                except ValueError:
                    w1 = np.nan
                out.append({"x_max": xm, "LB": lb, "lam": lam, "비교대상": nm,
                            "n": len(COMMON),
                            "DFL-MDD": round(x.mean(), 2),
                            "비교모델": round(y.mean(), 2),
                            "차이": round(diff.mean(), 2),
                            "Cohen d": round(abs(diff.mean() / diff.std(ddof=1)), 3),
                            "t": round(t, 2), "p (one-sided)": round(p1, 4),
                            **{f"유의({a:.2f})": sig(p1, a) for a in ALPHAS},
                            "Wilcoxon p": round(w1, 4)})
    return pd.DataFrame(out)

res_all = pd.concat([ttest_for(xm) for xm in sorted(DATA)], ignore_index=True)
out = f"{RESULT_DIR}/{N_STOCKS}_inds_h{HORIZON}_xmax_ttest.csv"
res_all.to_csv(out, index=False, encoding="utf-8-sig")
print(f"✓ 저장: {out}  ({len(res_all)}행)\n")

for xm in sorted(DATA):
    print(f"[x_max = {xm:g}]  α = 0.05")
    display(res_all[res_all.x_max == xm].pivot_table(
        index="비교대상", columns=["LB", "lam"],
        values="유의(0.05)", aggfunc="first"))